# 분포 파악 차트 — 히스토그램 · KDE · 박스플롯


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 연속형 수치 데이터의 분포를 히스토그램·KDE·박스플롯으로 시각화하고, 구간 수(`bins`), 확률 밀도, 사분위수(`Q1·Q2·Q3·IQR`)와 이상치 경계를 정확히 해석합니다.

## 처음 배우는 분을 위한 안내

히스토그램은 숫자 구간별 개수, 박스플롯은 중앙값과 가운데 50% 범위, KDE는 매끄럽게 그린 분포 곡선입니다. KDE의 세로축은 개수가 아닌 밀도입니다. bins는 구간 수, IQR은 Q3−Q1입니다. revenue는 매출, delivery_days는 배송일수입니다.

**학습 순서:** 문제 1 ~ 3은 필수 기초, KDE·사분위수 비교가 포함된 4 ~ 12는 선택 도전입니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 환경과 예제 데이터

Matplotlib 기본 폰트는 한글을 지원하지 않아 그래프 제목이나 축 이름이 깨질 수 있습니다. 시각화 코드 맨 앞에서 운영체제별 한글 폰트와 마이너스 기호(`axes.unicode_minus = False`)를 먼저 설정합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요.

```python
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 운영체제별 한글 폰트 및 마이너스 기호 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 학습용 가상 데이터 (외부 CSV 없이 바로 실행 가능)
online = pd.DataFrame({
    'order_id': range(101, 117),
    'revenue': [120, 150, 180, 200, 220, 240, 250, 260,
                280, 300, 320, 350, 380, 420, 450, 850],
    'delivery_days': [2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 7, 11]
})

offline = pd.DataFrame({
    'order_id': range(201, 217),
    'unit_price': [20, 22, 24, 25, 25, 26, 27, 28,
                   29, 30, 31, 32, 34, 35, 75, 90],
    'revenue': [180, 210, 230, 260, 290, 310, 330, 360,
                390, 410, 440, 480, 520, 550, 600, 920],
    'delivery_days': [1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 5, 8]
})

summary = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 15)],
    'membership_tier': ['Bronze', 'Bronze', 'Silver', 'Silver', 'Bronze', 'Silver', 'Gold',
                        'Silver', 'Gold', 'Gold', 'Silver', 'Bronze', 'Gold', 'Gold'],
    'avg_order_value': [40.0, 55.0, 60.0, 75.0, None, 80.0, 95.0,
                        110.0, 125.0, 140.0, 160.0, None, 210.0, 320.0],
    'total_spent': [180, 220, 250, 270, 290, 300, 310,
                    340, 360, 400, 450, 480, 1200, 1500]
})

sales = pd.DataFrame({
    'region': ['South'] * 10 + ['North'] * 10,
    'delivery_days': [1, 2, 2, 2, 3, 3, 3, 4, 4, 5,
                      3, 4, 4, 5, 5, 6, 6, 7, 8, 12],
    'revenue': [150, 180, 210, 230, 260, 290, 310, 340, 390, 450,
                200, 240, 280, 320, 350, 380, 420, 460, 520, 780]
})
```

## 분포 차트를 그리기 전에 확인할 것

숫자 목록이나 평균 하나만 보면 데이터가 한쪽으로 쏠려 있는지(비대칭), 봉우리가 여러 개인지(다봉 분포), 극단적인 이상치가 섞여 있는지 알기 어렵습니다. 분석 초기 단계에서 분포 차트를 먼저 확인해야 전처리 방향과 대표값(평균 vs 중앙값)을 올바르게 선택할 수 있습니다.

| 차트 종류 | 주요 함수 | Y축(또는 상자) 의미 | 강점 | 주의점 |
| --- | --- | --- | --- | --- |
| 히스토그램 | `plt.hist()`, `sns.histplot()` | 구간별 데이터 빈도(개수) | 실제 건수와 최빈 구간을 직관적으로 확인 | `bins` 개수에 따라 모양이 크게 달라짐 |
| KDE (커널 밀도) | `sns.kdeplot()`, `sns.displot(kde=True)` | 확률 밀도(전체 면적 = 1) | 부드러운 전체 윤곽 파악, 두 그룹 겹쳐 비교에 유리 | Y축이 실제 건수가 아님, 표본이 너무 적으면 과잉 추정 |
| 박스플롯 | `plt.boxplot()`, `sns.boxplot()` | 사분위수(Q1·Q2·Q3)와 수염·이상치 | 중앙값·퍼짐(IQR)·이상치 후보를 한눈에 요약 | 봉우리 개수(단봉/쌍봉) 등 세부 형태는 숨겨짐 |

## 히스토그램과 `bins` 설정의 영향

히스토그램은 연속형 수치 범위를 일정한 구간(`bins`)으로 나누고 각 구간에 속한 데이터 개수를 기둥 높이로 나타냅니다. 범주형 항목을 비교하는 막대그래프(`plt.bar`)와 달리 **막대 사이에 간격이 없으며**, X축은 연속된 수치 구간을 의미합니다.

| `bins` 설정 | 특징 | 권장 활용 |
| --- | --- | --- |
| 너무 적음 (예: 3 ~ 5) | 구간이 넓어져 세부 봉우리나 비대칭 패턴이 뭉개짐 | 소규모 표본의 거시적 요약 |
| 적정 수준 (예: 10 ~ 30) | 전체 분포 형태와 주요 빈도 구간을 균형 있게 표현 | 기본 탐색(`bins='auto'` 또는 15 ~ 30) |
| 너무 많음 (예: 50 ~ 100+) | 빈 구간과 잔물결(노이즈)이 많아져 전체 흐름 파악이 어려움 | 특정 경계값 밀집 여부 정밀 확인 |

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(online['revenue'], bins=4, color='steelblue', edgecolor='white')
axes[0].set_title('온라인 매출 분포 (bins=4)')
axes[0].set_xlabel('매출')
axes[0].set_ylabel('주문 수')

axes[1].hist(online['revenue'], bins=10, color='coral', edgecolor='white')
axes[1].set_title('온라인 매출 분포 (bins=10)')
axes[1].set_xlabel('매출')
axes[1].set_ylabel('주문 수')
plt.tight_layout()
plt.show()
```

Seaborn의 `sns.histplot(data=df, x='열', bins=..., hue='그룹열')` 또는 `sns.displot(..., kde=True)`를 사용하면 범주별 색상 구분(`hue`)과 밀도 곡선 결합을 한 줄로 처리할 수 있습니다. (과거 `sns.distplot`은 제거되었으므로 `histplot`이나 `displot`을 사용합니다.)

## KDE (커널 밀도 추정)와 그룹 분포 비교

KDE(Kernel Density Estimation)는 각 데이터 위치마다 매끄러운 커널(종 모양 곡선)을 놓아 합산함으로써 연속적인 분포 곡선을 추정합니다. 히스토그램처럼 `bins` 경계에 따라 모양이 급변하지 않으며, 두 개 이상의 그룹을 한 좌표계에 겹쳐 그릴 때 시각적 간섭이 적습니다.

- **Y축 해석 주의:** KDE의 Y축은 '고객 수'나 '건수'가 아니라 **밀도(Density)**입니다. 곡선 아래 전체 면적의 합이 `1`이 되도록 정규화되어 있으므로, 곡선이 높다는 것은 해당 구간에 데이터가 **상대적으로 조밀하게 몰려 있음**을 뜻합니다.
- `fill=True` 옵션을 주면 곡선 아래 영역을 반투명하게 채워 두 그룹의 중심 위치 이동과 겹치는 범위를 쉽게 비교할 수 있습니다.

```python
plt.figure(figsize=(7, 4))
sns.kdeplot(online['revenue'], label='Online', fill=True)
sns.kdeplot(offline['revenue'], label='Offline', fill=True)
plt.xlabel('주문 매출')
plt.ylabel('밀도 (Density)')
plt.title('채널별 주문 매출 분포 비교 (KDE)')
plt.legend()
plt.show()
```

## 박스플롯(Box Plot)과 사분위수 · IQR 이상치 기준

박스플롯은 다섯 가지 요약 통계량과 잠재적 이상치를 한 번에 표현합니다. 평균이 극단값에 끌려가는 상황에서 중심 위치(중앙값)와 중간 50%의 퍼짐 정도를 안정적으로 보여줍니다.

| 구성 요소 | 통계적 의미 | 계산식 / 위치 |
| --- | --- | --- |
| 박스 하단 | 제1사분위수 (`Q1`, 하위 25%) | `series.quantile(0.25)` |
| 박스 내부 중앙선 | 중앙값 (`Q2`, Median, 50%) | `series.median()` 또는 `quantile(0.50)` |
| 박스 상단 | 제3사분위수 (`Q3`, 상위 25% 경계 = 75%) | `series.quantile(0.75)` |
| 박스 길이 | 사분위범위 (`IQR`, 중간 50% 데이터 폭) | `IQR = Q3 - Q1` |
| 수염 (Whisker) | 정상 범위 경계 안쪽의 최솟값·최댓값 | 하한 `Q1 - 1.5×IQR` ~ 상한 `Q3 + 1.5×IQR` 이내 |
| 수염 바깥 점 (●) | 잠재적 이상치 (Outlier 후보) | `값 < Q1 - 1.5×IQR` 또는 `값 > Q3 + 1.5×IQR` |

```python
q1 = offline['unit_price'].quantile(0.25)
q2 = offline['unit_price'].median()
q3 = offline['unit_price'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outliers = offline.loc[
    (offline['unit_price'] < lower_fence) | (offline['unit_price'] > upper_fence),
    'unit_price'
]
print(f"Q1={q1}, 중앙값={q2}, Q3={q3}, IQR={iqr}")
print(f"정상 범위 경계: [{lower_fence}, {upper_fence}] / 이상치 후보: {outliers.tolist()}")
```

수염 바깥에 찍힌 점은 **'자동 삭제 대상'이 아니라 '추가 확인이 필요한 후보'**입니다. 입력 오류인지, 실제 VIP 고객이나 특수 대량 주문인지 도메인 맥락을 먼저 확인해야 합니다.

## 세 가지 분포 차트 나란히 비교하기

같은 데이터라도 어떤 차트로 보느냐에 따라 눈에 띄는 정보가 다릅니다. 실무 탐색에서는 `plt.subplots(1, 3)`으로 세 차트를 나란히 배치해 상호 보완적으로 해석합니다.

```python
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(online['revenue'], bins=6, color='steelblue', edgecolor='white')
axes[0].set_title('히스토그램 (구간별 빈도)')
axes[0].set_xlabel('매출')
axes[0].set_ylabel('건수')

sns.kdeplot(online['revenue'], ax=axes[1], color='coral', fill=True)
axes[1].set_title('KDE (전체 분포 형태·비대칭)')
axes[1].set_xlabel('매출')

axes[2].boxplot(online['revenue'])
axes[2].set_title('박스플롯 (중앙값·IQR·이상치)')
axes[2].set_ylabel('매출')

plt.tight_layout()
plt.show()
```

## 선택 도전: 다섯 개 금액의 중심과 꼬리 비교

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_values = pd.Series([10,12,12,13,50],name='amount')
```

### 진행 순서

1. 평균과 중앙값을 출력합니다.
2. bins=4인 히스토그램을 그립니다. X축 Amount, Y축 Count, 제목 Small Amount Distribution을 붙입니다.
3. 50이 평균을 얼마나 끌어올렸는지 설명합니다.

**완료 기준:** 평균 19.4, 중앙값 12, 히스토그램의 빈도 합계 5. 실제 금액 단위는 이 가상 예제에서 지정하지 않음.

### 선택 도전 풀이

```python
import matplotlib.pyplot as plt
print(extra_values.mean(),extra_values.median())
counts,edges,_=plt.hist(extra_values,bins=4,edgecolor='white')
plt.xlabel('Amount');plt.ylabel('Count');plt.title('Small Amount Distribution');plt.show()
assert counts.sum()==5 and extra_values.median()==12
```

## 복습 순서

1. 한글 폰트 설정과 준비 데이터의 행 수·결측치 유무를 먼저 점검합니다.
2. 히스토그램에서 `bins` 값을 바꾸어 보며 최빈 구간과 분포 모양 변화를 관찰합니다.
3. 두 그룹 이상의 분포를 겹쳐 볼 때 KDE(`fill=True`)가 주는 이점과 Y축(밀도)의 의미를 설명합니다.
4. `quantile(0.25)`, `quantile(0.75)`로 `IQR`과 `1.5×IQR` 경계를 직접 계산해 박스플롯의 이상치 점과 일치하는지 검증합니다.
5. `_Practice.ipynb`의 12문제를 직접 코드로 작성하고 해석을 기록합니다.